# Ridge, Lasso, and Elastic Net

### Short, beginner-friendly notes and Python practice

**Big idea:** These methods help a regression model avoid memorizing its training examples. They do this by discouraging very large feature weights.

By the end, you should be able to explain regularization, compare the three methods, and understand a simple coefficient diagram.

## 1. Remember linear regression

A regression model predicts a number by adding feature values multiplied by weights:

**prediction = intercept + weight₁ × feature₁ + weight₂ × feature₂ + ...**

Example: predict a test score using study hours, sleep, and practice questions. A weight tells how much the prediction changes when that feature changes and other features stay the same.

The model learns weights that make predictions close to the real answers. A common error measure is the average of squared prediction mistakes.

## 2. Why use regularization?

A student can memorize practice questions but struggle with a new question. A model can do the same: fit its training data very closely but make poor predictions on new data. This is **overfitting**.

Regularization adds an extra cost when the model uses very large weights. The model tries to:

1. Make predictions close to the training answers.
2. Keep the feature weights reasonably small.

It is a gentle nudge, not a rule that forces every model to be simple. We check whether it helps using data the model did not train on.

## 3. The penalty idea

Ordinary linear regression mainly minimizes **prediction error**. Regularized regression minimizes:

**total cost = prediction error + penalty for large weights**

The intercept (the starting value) is usually not part of the penalty. The penalty amount is controlled by **lambda (λ)**. In scikit-learn, this setting is called **alpha**.

- **λ / alpha = 0:** no regularization; this is ordinary linear regression.
- **Small λ / alpha:** a light nudge toward smaller weights.
- **Large λ / alpha:** a strong nudge; the model may become too simple.

Penalty formulas use slightly different scaling in different books and software. The key idea is that alpha controls how strongly we discourage large weights.

## 4. Ridge regression (L2)

Ridge adds the **squares** of the weights to the cost:

**cost = prediction error + λ × (weight₁² + weight₂² + ...)**

### In simple words

Ridge pulls all feature weights toward zero. A weight usually gets smaller, but does **not** become exactly zero. Ridge keeps all features, but gives them less influence.

**Example:** A weight might change from 0.50 to 0.38. That feature still matters, but its effect on the prediction is smaller.

**Good when:** Many features may each help a little, especially when features overlap or are strongly related.

## 5. Lasso regression (L1)

Lasso adds the **absolute values** of the weights to the cost:

**cost = prediction error + λ × (|weight₁| + |weight₂| + ...)**

Lasso also pulls weights toward zero. It can pull some weights **all the way to zero**. A zero weight means the model is not using that feature.

**Example:** If the weight for “number of stickers on a notebook” becomes 0, the model ignores that feature when predicting a test score.

**Good when:** You have many features and want a smaller set of features. If several features contain nearly the same information, Lasso's exact feature choices can be unstable.

## 6. Elastic Net (L1 + L2)

Elastic Net mixes the Lasso and Ridge penalties:

**cost = prediction error + L1 penalty + L2 penalty**

It can make some weights zero like Lasso, while shrinking the other weights like Ridge.

**Good when:** You have many features and some are strongly related. It is a middle ground between Ridge and Lasso.

In scikit-learn, **l1_ratio** controls the mix: 0 is Ridge-like, 1 is Lasso-like, and a value between 0 and 1 mixes both.

## 7. Quick comparison

| Method | Penalty | What happens to weights? | Memory clue |
|---|---|---|---|
| Ordinary linear regression | None | Can be large | Fits error only |
| Ridge (L2) | Squared weights | Smaller; usually not zero | Keeps every feature |
| Lasso (L1) | Absolute weights | Some can become zero | Can switch features off |
| Elastic Net | L1 + L2 | Shrinks; some can become zero | Mix of Ridge + Lasso |

**One-line revision:** Ridge shrinks; Lasso can shrink and switch features off; Elastic Net does some of both.

## 8. What happens when lambda gets bigger?

A bigger lambda (or scikit-learn alpha) means a stronger penalty, so the model is pushed to use smaller weights. Ridge weights move closer to zero. With Lasso and Elastic Net, some weights may reach zero.

If the penalty is **too strong**, the model may become too simple and miss real patterns. This is called **underfitting**. We do not pick alpha just because bigger is better; we compare choices using cross-validation and a separate test set.

A perfect training score can be a warning sign, but does not prove overfitting by itself. Compare training results with validation or test results.

## 9. Important: put features on a similar scale

One feature might be measured in years (values around 1–10) and another in dollars (values around 1,000–100,000). Their weight sizes are hard to compare fairly. Regularization could penalize them unevenly just because of their units.

We usually **standardize** features first so they are on a comparable scale. The pipeline below learns scaling from training data only. This prevents information from the test set leaking into training.

In scikit-learn, the intercept is not penalized by Ridge, Lasso, or Elastic Net.

## 10. Practice: make a small pretend dataset

We will predict a student's score using study hours, practice questions, sleep hours, and phone time. These are made-up examples, so no download is needed.

The score is created from the four features plus a little random noise. In a real project, use measured data instead.

In [ ]:
import numpy as np  # Make numbers and random examples.
import pandas as pd  # Put the example data into a readable table.
import matplotlib.pyplot as plt  # Draw the coefficient diagram.

from sklearn.model_selection import train_test_split  # Keep some data aside for a fair test.
from sklearn.pipeline import make_pipeline  # Keep scaling and regression together.
from sklearn.preprocessing import StandardScaler  # Put features on similar scales.
from sklearn.linear_model import Ridge, Lasso, ElasticNet  # The three models.
from sklearn.metrics import mean_squared_error, r2_score  # Measure test predictions.

### What the import lines mean

- numpy makes the pretend measurements and random noise.
- pandas organizes the measurements into named columns.
- matplotlib draws the picture.
- train_test_split saves unseen rows for a fair check.
- StandardScaler makes feature scales comparable.
- make_pipeline makes sure scaling happens before regression.
- Ridge, Lasso, and ElasticNet are the three models we are studying.
- The metrics help us judge predictions on the unseen test rows.

In [ ]:
rng = np.random.default_rng(7)  # Use the same random examples every time.
rows = 240  # Make 240 pretend students.

study_hours = rng.uniform(0, 8, rows)  # Each studies between 0 and 8 hours.
practice_questions = 2 * study_hours + rng.normal(0, 3, rows)  # Practice and study overlap a little.
sleep_hours = rng.uniform(4, 10, rows)  # Sleep between 4 and 10 hours.
phone_hours = rng.uniform(0, 7, rows)  # Phone use between 0 and 7 hours.

noise = rng.normal(0, 5, rows)  # Real results are not perfectly predictable.
score = (50 + 3 * study_hours + 1.2 * practice_questions
         + 1.5 * sleep_hours - 2 * phone_hours + noise)  # Make a pretend score.

data = pd.DataFrame({  # Give each group of values a clear column name.
    "study_hours": study_hours,
    "practice_questions": practice_questions,
    "sleep_hours": sleep_hours,
    "phone_hours": phone_hours,
    "score": score,
})

data.head()  # Show the first five pretend students.

### Read the data-making code

- default_rng(7) makes the random data repeatable.
- uniform picks values from a range; normal adds natural-looking variation.
- practice_questions is related to study_hours, so the two features overlap a little.
- Study, practice, and sleep raise this pretend score; phone hours lower it.
- DataFrame puts the columns together, and head() previews five rows.

These numbers are only for learning. They are not a rule about real students.

## 11. Save some data for a fair test

The model learns from the **training set**. The **test set** stays unseen until we check predictions. It is like learning with practice questions, then trying a fresh quiz.

In [ ]:
feature_names = ["study_hours", "practice_questions", "sleep_hours", "phone_hours"]  # Model inputs.
X = data[feature_names]  # X holds the clues/features.
y = data["score"]  # y holds the answer to predict.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)  # Learn from 75%; keep 25% unseen for the test.

### Read the split code

- X holds the four clues; y holds the scores.
- test_size=0.25 saves one quarter of the rows for the final check.
- random_state=7 makes the split repeatable.
- The pipeline fits the scaler using X_train only, not the test rows.

## 12. Train the three models and compare test predictions

Every method uses the same training and test rows. These alpha values are starting examples, not guaranteed best settings. A real project should tune them with cross-validation.

In [ ]:
models = {  # Give each method a name and a starting penalty strength.
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.2, max_iter=20_000)),
    "Elastic Net": make_pipeline(StandardScaler(), ElasticNet(alpha=0.2, l1_ratio=0.5, max_iter=20_000)),
}

results = []  # Save each model's test scores here.
coefficient_rows = []  # Save each model's feature weights here.

for name, pipeline in models.items():  # Repeat for Ridge, Lasso, and Elastic Net.
    pipeline.fit(X_train, y_train)  # Learn using training rows only.
    predictions = pipeline.predict(X_test)  # Guess scores for unseen rows.
    regressor = pipeline.steps[-1][1]  # Get the fitted regression model.

    results.append({  # Keep two test measurements.
        "model": name,
        "test_RMSE": mean_squared_error(y_test, predictions) ** 0.5,
        "test_R2": r2_score(y_test, predictions),
    })
    coefficient_rows.append(regressor.coef_)  # Save this model's feature weights.

results_table = pd.DataFrame(results).round(3)  # Make a neat comparison table.
results_table

### Read the training and score code

- A pipeline means: first standardize, then fit the regression model.
- fit is the learning step. It sees training data only.
- predict asks the trained model to guess scores for hidden test rows.
- **RMSE** is a typical prediction mistake, in score points. Smaller is better.
- **R²** compares the model with guessing the average score. Closer to 1 is usually better. Negative test R² means it did worse than that simple guess.
- This one test split is useful for practice, but does not prove one method is always best. Use cross-validation to choose alpha.

## 13. Compare the feature weights

Inputs were standardized, so these weights are on a comparable scale. A positive weight pushes the prediction up; a negative weight pushes it down. A zero weight means Lasso or Elastic Net is not using that feature in this fitted model.

In [ ]:
coefficient_table = pd.DataFrame(  # Make a table with model and feature names.
    coefficient_rows,
    index=models.keys(),
    columns=feature_names,
).round(3)

coefficient_table

### How to read the weight table

- Look across one row to see the four weights for one method.
- Compare a feature's weight across rows to see how each penalty changes it.
- A smaller weight means a smaller effect in this fitted model; it does not alone prove a feature is useless.
- Lasso or Elastic Net may show a zero weight, meaning that model switched the feature off. Rounded values can display as 0.0 even if they are only close to zero.
- Weights can change when the data or alpha changes.

## 14. Visual: increase the penalty and watch weights shrink

This picture refits each method with stronger and stronger alpha. It shows how every feature's weight changes. All three panels use the same training data and standardize features.

In [ ]:
alphas = np.logspace(-2, 3, 28)  # Try strengths from 0.01 up to 1000.
model_types = ["Ridge", "Lasso", "Elastic Net"]  # One picture panel per method.
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)  # Make three side-by-side plots.

for ax, name in zip(axes, model_types):  # Make one coefficient plot at a time.
    coefficient_path = []  # Store weights for every alpha.
    for alpha in alphas:  # Try one penalty strength at a time.
        if name == "Ridge":
            model = Ridge(alpha=alpha)
        elif name == "Lasso":
            model = Lasso(alpha=alpha, max_iter=5_000)
        else:
            model = ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=5_000)

        pipeline = make_pipeline(StandardScaler(), model)  # Scale, then fit.
        pipeline.fit(X_train, y_train)  # Use training rows only.
        coefficient_path.append(pipeline.steps[-1][1].coef_)  # Save this fit's weights.

    coefficient_path = np.array(coefficient_path)  # Make a table for plotting.
    for feature_index, feature in enumerate(feature_names):  # Draw a line per feature.
        ax.plot(alphas, coefficient_path[:, feature_index], label=feature)

    ax.set_xscale("log")  # Alpha covers a wide range, so use a log scale.
    ax.set_title(name)
    ax.set_xlabel("alpha (penalty strength)")
    ax.axhline(0, color="black", linewidth=0.8)  # Make zero weight easy to see.
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("feature weight after standardizing")
axes[-1].legend(loc="best", fontsize=8)
fig.suptitle("Stronger penalty pulls feature weights toward zero")
fig.tight_layout()
plt.show()

### Understand the diagram

- **Left to right:** alpha gets larger, so the penalty gets stronger.
- **Up and down:** the size and direction of a feature's weight.
- **Ridge:** lines move toward zero, but generally do not land exactly on zero.
- **Lasso:** lines can land exactly on zero, switching some features off.
- **Elastic Net:** can shrink weights and can set some to zero too.

The exact shape depends on the data. Focus on the main message: stronger penalty means smaller weights.

## 15. Choosing a penalty in a real project

Do not guess that one alpha is best. Try several values with **cross-validation**: train on some pieces of the training data and check on the remaining piece. Choose alpha that works well across those checks. Then evaluate the chosen model once on the untouched test set.

Scikit-learn tools such as GridSearchCV, RidgeCV, LassoCV, and ElasticNetCV can help. Keep the scaler inside the pipeline so each check learns scaling from its training portion only.

## Quick revision card

- **Problem:** A model can memorize training data and struggle with new examples.
- **Regularization:** Adds a cost for large weights to discourage an overly complex fit.
- **Lambda / alpha:** Controls penalty strength; a stronger penalty usually means smaller weights.
- **Ridge = L2:** Squares weights; shrinks them and usually keeps all features.
- **Lasso = L1:** Uses absolute weights; can make some weights zero.
- **Elastic Net:** Combines L1 and L2.
- **Before fitting:** Standardize features; learn scaling from training data only.
- **Choose alpha:** Use cross-validation; keep a separate test set for the final check.

### Say it in one breath

**Ridge makes weights smaller, Lasso can switch features off, and Elastic Net combines both ideas.**